#### Validation: Ensures LLM outputs match predefined data schemas. This component provides schema validation and structured data parsing to guarantee consistent data formats for downstream code.




#### NOTE: i am using github openai for this example.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from pydantic import BaseModel

In [3]:
TOKEN = os.getenv('GITHUB_TOKEN')
ENDPOINT = os.getenv('GITHUB_ENDPOINT')
MODEL = os.getenv('GITHUB_MODEL_NAME')

client = OpenAI(
    base_url=ENDPOINT,
    api_key=TOKEN,
)

In [6]:
class TaskResult(BaseModel):
    task: str
    result: str
    priority: int

def structured_intelligence(promt: str) -> TaskResult:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system", 
                "content": (
                    "You are a helpful assistant. "
                    "You must reply ONLY with a valid JSON object matching this schema: "
                    '{"task": str, "result": str, "priority": int}'
                ),
            },
            {"role": "user", "content": promt}
        ],
        max_tokens=1000,
        temperature=0.2,
    )
    
    content = response.choices[0].message.content
    
    return TaskResult.model_validate_json(content)

In [7]:
result = structured_intelligence(
    "I need to complete the project presentation by Friday, it's high priority"
)
print("Structured Output:")
print(result.model_dump_json(indent=2))
print(f"Extracted task: {result.task}")

Structured Output:
{
  "task": "Complete the project presentation by Friday",
  "result": "Project presentation will be ready by Friday",
  "priority": 3
}
Extracted task: Complete the project presentation by Friday
